In [1]:
from glob import glob

for g in glob("../data/*.pdf"):
    print(g)

../data\2040_seoul_plan.pdf
../data\OneNYC_2050_StrategicPlan.pdf


read_pdf_and_split_text 함수 만들기


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """
    주어진 PDF 파일을 읽고 텍스트를 분할합니다.
    매개변수:
      pdf_path (str): PDF 파일의 경로.
      chunk_size (int, 선택적): 각 텍스트 청크의 크기, 기본값은 1000 입니다.
      chunk_overlap (int, 선택적): 청크 간의 중첩 크기. 기본값은 100 입니다.
    반환값:
      list: 분할된 텍스트 청크의 리스트.
    """

    print(f"PDF: {pdf_path} -----------------------------------")

    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )

    splits = text_splitter.split_documents(data_from_pdf)

    print(f"Number of splits: {len(splits)}\n")
    return splits

C:\Users\bny64\AppData\Local\Temp\ipykernel_10384\70338277.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from glob import glob
import os
import time
import dotenv

dotenv.load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# 임베딩 모델 선언
embedding = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2", google_api_key=GEMINI_API_KEY
)

persist_directory = "../chroma_store"

if os.path.exists(persist_directory):
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory, embedding_function=embedding
    )

else:
    print("Creating new Chroma store")
    # 429 Resource Exhausted 방지를 위해 청크 단위로 나누어 인덱싱을 수행합니다.
    vectorstore = None
    batch_size = 40
    
    for g in glob("../data/*.pdf"):
        chunks = read_pdf_and_split_text(g)
        for i in range(0, len(chunks), batch_size):
            print(f"Adding documents from {i} to {min(i + batch_size, len(chunks))} for {g}...")
            if vectorstore is None:
                # 첫 번째 청크로 vectorstore 초기화
                vectorstore = Chroma.from_documents(
                    documents=chunks[i : i + batch_size],
                    embedding=embedding,
                    persist_directory=persist_directory,
                )
            else:
                vectorstore.add_documents(documents=chunks[i : i + batch_size])
            time.sleep(60)
    print("Chroma store creation completed!")


Creating new Chroma store
PDF: ../data\2040_seoul_plan.pdf -----------------------------------
Number of splits: 361

Adding documents from 0 to 40 for ../data\2040_seoul_plan.pdf...
Adding documents from 40 to 80 for ../data\2040_seoul_plan.pdf...
Adding documents from 80 to 120 for ../data\2040_seoul_plan.pdf...
Adding documents from 120 to 160 for ../data\2040_seoul_plan.pdf...
Adding documents from 160 to 200 for ../data\2040_seoul_plan.pdf...
Adding documents from 200 to 240 for ../data\2040_seoul_plan.pdf...
Adding documents from 240 to 280 for ../data\2040_seoul_plan.pdf...
Adding documents from 280 to 320 for ../data\2040_seoul_plan.pdf...
Adding documents from 320 to 360 for ../data\2040_seoul_plan.pdf...
Adding documents from 360 to 361 for ../data\2040_seoul_plan.pdf...
PDF: ../data\OneNYC_2050_StrategicPlan.pdf -----------------------------------
Number of splits: 176

Adding documents from 0 to 40 for ../data\OneNYC_2050_StrategicPlan.pdf...
Adding documents from 40 to 80 

In [4]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

chunks = retriever.invoke("서울 온실가스 저감 계획")

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)

{'creationdate': '2023-02-14T11:05:36+09:00', 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'moddate': '2023-02-14T18:21:42+09:00', 'page': 70, 'author': '', 'creator': 'PScript5.dll Version 5.2.2', 'title': '', 'page_label': '71', 'source': '../data\\2040_seoul_plan.pdf', 'total_pages': 272}
- 도시 기반시설 부문(건물): 무공해 빌딩 확대
- 도시 기반시설 부문(교통): 무공해 차량(ZEV) 보급 촉진
- 자원/산업 부문: 3R(줄이기, 재사용, 재활용), 플라스틱, 음식물쓰레기, HFC배출량 감소
- 기후변화 적응: 기후변화 적응 조치 강화
- 거버넌스: 기업(민간), 지자체, 주요 도시 등과 협력, 재생에너지 사업 투자 촉진
8) https://fpcj.jp/en/prlisting/tokyo_20211012/
9) https://zenbird.media/zero-emissions-tokyo-an-ambitious-climate-change-strategy/
{'title': '', 'source': '../data\\2040_seoul_plan.pdf', 'author': '', 'creator': 'PScript5.dll Version 5.2.2', 'page': 151, 'total_pages': 272, 'producer': 'Acrobat Distiller 9.0.0 (Windows)', 'page_label': '152', 'creationdate': '2023-02-14T11:05:36+09:00', 'moddate': '2023-02-14T18:21:42+09:00'}
3. 부문별 전략계획
3.1 추진위원회 회의 결과
3.2 서울시 관련 실·국·본부 의견
{'moddate': '2023-02-14T18:21: